In [8]:
# 0. import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 시각화 설정
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

In [9]:
# 1. dataloading
df = pd.read_csv('/Users/othree/Cognitive Reserve Modeling/Data/ADNI_master_merged_12-17-2025.csv')

print("="*50)
print("데이터 로드 완료!")
print("="*50)
print(f"전체 행 수: {df.shape[0]:,}")
print(f"전체 열 수: {df.shape[1]:,}")
print(f"총 데이터 포인트: {df.shape[0] * df.shape[1]:,}")

데이터 로드 완료!
전체 행 수: 4,508
전체 열 수: 159
총 데이터 포인트: 716,772


In [10]:
# 4. Remove rows with any missing values and create clean versions
thresholds = [3, 5, 10, 30, 50]

print("="*80)
print("결측치가 있는 행 제거")
print("="*80)

for threshold in thresholds:
    # Read the previously created CSV
    input_filename = f'/Users/othree/Cognitive Reserve Modeling/Data/ADNI_merged_missing_lt{threshold}pct.csv'
    df_filtered = pd.read_csv(input_filename)
    
    original_rows = len(df_filtered)
    
    # Remove rows with any missing values
    df_clean = df_filtered.dropna()
    
    clean_rows = len(df_clean)
    removed_rows = original_rows - clean_rows
    
    # Save clean version
    output_filename = f'/Users/othree/Cognitive Reserve Modeling/Data/ADNI_merged_missing_lt{threshold}pct_clean.csv'
    df_clean.to_csv(output_filename, index=False)
    
    print(f"\n< {threshold}%:")
    print(f"  Original: {original_rows:,} rows")
    print(f"  Clean: {clean_rows:,} rows")
    print(f"  Removed: {removed_rows:,} rows ({removed_rows/original_rows*100:.2f}%)")
    print(f"  Saved: ADNI_merged_missing_lt{threshold}pct_clean.csv")

print("\n" + "="*80)
print("All clean CSV files created successfully!")
print("="*80)

결측치가 있는 행 제거

< 3%:
  Original: 4,508 rows
  Clean: 4,126 rows
  Removed: 382 rows (8.47%)
  Saved: ADNI_merged_missing_lt3pct_clean.csv

< 5%:
  Original: 4,508 rows
  Clean: 3,714 rows
  Removed: 794 rows (17.61%)
  Saved: ADNI_merged_missing_lt5pct_clean.csv

< 10%:
  Original: 4,508 rows
  Clean: 3,564 rows
  Removed: 944 rows (20.94%)
  Saved: ADNI_merged_missing_lt10pct_clean.csv

< 30%:
  Original: 4,508 rows
  Clean: 1,500 rows
  Removed: 3,008 rows (66.73%)
  Saved: ADNI_merged_missing_lt30pct_clean.csv

< 50%:
  Original: 4,508 rows
  Clean: 313 rows
  Removed: 4,195 rows (93.06%)
  Saved: ADNI_merged_missing_lt50pct_clean.csv

All clean CSV files created successfully!


In [11]:
# 3. Create CSV files with filtered columns
base_columns = ['New_Path', 'Image Data ID', 'Subject', 'Group-RS:InitialDX', 'Sex', 'Age', 
                'Visit', 'Description', 'Acq Date', 'RS:DX_fill', 'DX2', 'RID', 
                'COLPROT', 'ORIGPROT', 'PTID', 'SITE', 'VISCODE', 'EXAMDATE_x', 
                'DX_bl', 'AGE', 'PTGENDER', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT', 
                'PTMARRY', 'Years_bl', 'Month_bl']

missing_counts = df.isnull().sum()
total_rows = len(df)
missing_pct = (missing_counts / total_rows * 100).sort_values()

thresholds = [3, 5, 10, 30, 50]

for threshold in thresholds:
    # Get columns with missing < threshold%
    missing_cols = [col for col in missing_pct[missing_pct < threshold].index 
                    if col not in base_columns]
    
    # Combine: base columns first, then missing columns
    selected_cols = base_columns + missing_cols
    
    # Filter only columns that exist in df
    selected_cols = [col for col in selected_cols if col in df.columns]
    
    # Create filtered dataframe
    df_filtered = df[selected_cols]
    
    # Save to CSV
    output_filename = f'/Users/othree/Cognitive Reserve Modeling/Data/ADNI_merged_missing_lt{threshold}pct.csv'
    df_filtered.to_csv(output_filename, index=False)
    
    print(f"Saved: ADNI_merged_missing_lt{threshold}pct.csv ({len(selected_cols)} columns)")

print("\nAll CSV files created successfully!")

Saved: ADNI_merged_missing_lt3pct.csv (66 columns)
Saved: ADNI_merged_missing_lt5pct.csv (70 columns)
Saved: ADNI_merged_missing_lt10pct.csv (72 columns)
Saved: ADNI_merged_missing_lt30pct.csv (82 columns)
Saved: ADNI_merged_missing_lt50pct.csv (114 columns)

All CSV files created successfully!


In [12]:
# 2. Missing value analysis by percentage
missing_counts = df.isnull().sum()
total_rows = len(df)
missing_pct = (missing_counts / total_rows * 100).sort_values()

thresholds = [3, 5, 10, 30, 50]

print("="*80)
print("결측치 분석 (퍼센트 기준)")
print("="*80)

for threshold in thresholds:
    cols = [col for col in missing_pct[missing_pct < threshold].index]
    
    print(f"\n{'='*80}")
    print(f"결측치 < {threshold}% 컬럼들 ({len(cols)}개):")
    print(f"{'='*80}")
    
    for col in cols:
        count = missing_counts[col]
        pct = missing_pct[col]
        print(f"  - {col:30s}: {count:4d}개 ({pct:5.2f}%)")

print(f"\n{'='*80}")
print("요약:")
print(f"{'='*80}")
for threshold in thresholds:
    count = (missing_pct < threshold).sum()
    print(f"결측치 < {threshold:2d}%: {count:3d}개 컬럼")

결측치 분석 (퍼센트 기준)

결측치 < 3% 컬럼들 (66개):
  - New_Path                      :    0개 ( 0.00%)
  - PTRACCAT                      :    0개 ( 0.00%)
  - PTMARRY                       :    0개 ( 0.00%)
  - FSVERSION                     :    0개 ( 0.00%)
  - IMAGEUID                      :    0개 ( 0.00%)
  - ICV                           :    0개 ( 0.00%)
  - DX                            :    0개 ( 0.00%)
  - EXAMDATE_bl                   :    0개 ( 0.00%)
  - CDRSB_bl                      :    0개 ( 0.00%)
  - ADASQ4_bl                     :    0개 ( 0.00%)
  - LDELTOTAL_BL                  :    0개 ( 0.00%)
  - mPACCdigit_bl                 :    0개 ( 0.00%)
  - mPACCtrailsB_bl               :    0개 ( 0.00%)
  - Years_bl                      :    0개 ( 0.00%)
  - Month_bl                      :    0개 ( 0.00%)
  - Month                         :    0개 ( 0.00%)
  - M                             :    0개 ( 0.00%)
  - update_stamp_x                :    0개 ( 0.00%)
  - PTETHCAT                      :    0개 ( 0